# Intraday exploration: cost versus move, and volatility by hour (Phase 8)

Phase 8 of the intraday track (see ROADMAP.md). BTC/USDT only, using the existing 15-minute data; no new data is downloaded. This notebook is descriptive: it defines no setup and makes no trading decision.

**Questions**
1. Cost versus move: for holding periods of 15 minutes, 1 hour, 4 hours and 1 day, how large is the typical move, and how large is the round-trip cost (0.30%) relative to it? Overall and by year.
2. Intraday volatility: how does the standard deviation of 15-minute log returns vary by hour of day (UTC), overall and for 2017–2023 versus 2024 onward?

**Definitions (fixed before any result was computed)**
- Periods are aligned to UTC: 15-minute periods start on the quarter hour, 1-hour periods on the hour, 4-hour periods at 00:00, 04:00, …, 20:00, and 1-day periods at 00:00.
- The return of a period is log(open at period end / open at period start), using 15-minute candle opens. Periods follow each other without overlap.
- A period is dropped if the 15-minute candle at its start or its end is missing (the data has gaps).
- Typical move = sample standard deviation of these returns. Cost-to-move ratio = 0.30% / typical move.
- A return's year and hour of day are those of its period start (UTC).
- The 15-minute returns in question 2 are the 15-minute period returns from question 1.

## Step 0: Imports and settings

`COST_PER_SIDE` is the project cost (0.10% fee + 0.05% slippage); the round trip is twice that.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from cryptosense.data import load_processed, timeframe_to_ms

DATA_DIR = Path.cwd().parent / "data"
SYMBOL = "BTC/USDT"
COST_PER_SIDE = 0.0015
ROUND_TRIP = 2 * COST_PER_SIDE
PERIODS = ["15m", "1h", "4h", "1d"]
SPLIT = pd.Timestamp("2024-01-01", tz="UTC")

## Step 1: Load the 15-minute data

The existing processed file is loaded; no download.

In [2]:
m15 = load_processed(SYMBOL, DATA_DIR, timeframe="15m")
candle_ms = timeframe_to_ms("15m")
steps = m15["timestamp"].diff()
missing = int((steps[steps > candle_ms] // candle_ms - 1).sum())

print(f"candles: {len(m15):,}")
print(f"range:   {m15.index[0]} -> {m15.index[-1]}")
print(f"missing candles inside the range: {missing:,}")

# 15-minute opens looked up by integer candle timestamp (ms, UTC)
open_15m = m15.set_index("timestamp")["open"]

candles: 321,296
range:   2017-08-17 04:00:00+00:00 -> 2026-10-21 21:00:00+00:00
missing candles inside the range: 565


## Step 2: Non-overlapping period returns

For each holding period, returns on the UTC-aligned grid, open to open. Periods with a missing start or end candle are dropped and counted.

In [3]:
def period_returns(period_ms):
    """Log returns of consecutive UTC-aligned periods, open to open; NaN where an endpoint candle is missing."""
    first_ts, last_ts = int(open_15m.index[0]), int(open_15m.index[-1])
    starts = np.arange(-(-first_ts // period_ms) * period_ms, last_ts - period_ms + 1, period_ms)
    p_start = open_15m.reindex(starts).to_numpy()
    p_end = open_15m.reindex(starts + period_ms).to_numpy()
    index = pd.to_datetime(starts, unit="ms", utc=True)
    return pd.Series(np.log(p_end / p_start), index=index, name="log_return")


returns = {}
rows = []
for period in PERIODS:
    r = period_returns(timeframe_to_ms(period))
    returns[period] = r.dropna()
    rows.append({"period": period, "periods_on_grid": len(r), "dropped (missing candle)": int(r.isna().sum()),
                 "returns_used": int(r.notna().sum()), "first_start": returns[period].index[0],
                 "last_start": returns[period].index[-1]})
pd.DataFrame(rows).set_index("period")

,periods_on_grid,dropped (missing candle),returns_used,first_start,last_start
period,,,,,
15m,321860,598,321262,2017-08-17 04:00:00+00:00,2026-10-21 20:45:00+00:00
1h,80465,173,80292,2017-08-17 04:00:00+00:00,2026-10-21 20:00:00+00:00
4h,20116,53,20063,2017-08-17 04:00:00+00:00,2026-10-21 16:00:00+00:00
1d,3351,2,3349,2017-08-18 00:00:00+00:00,2026-10-20 00:00:00+00:00


## Step 3: Cost versus move — overall and by year

For each period: number of returns (n), typical move (standard deviation of log returns, in %), and the ratio of the 0.30% round-trip cost to the typical move. 2017 starts on 2017-08-17 and the last year is partial (to the end of the data).

In [4]:
def cost_vs_move(r):
    """n, typical move (%) and round-trip cost / typical move for one set of returns."""
    move = r.std()
    return {"n": len(r), "typical_move_%": 100 * move, "cost_to_move": ROUND_TRIP / move}


blocks = {}
for period, r in returns.items():
    by_year = {str(year): cost_vs_move(group) for year, group in r.groupby(r.index.year)}
    blocks[period] = pd.DataFrame({"overall": cost_vs_move(r), **by_year}).T

cost_table = pd.concat(blocks, axis=1)
cost_table.columns.names = ["period", "metric"]
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print(cost_table.to_string(formatters={
    col: (lambda v: f"{v:,.0f}") if col[1] == "n" else (lambda v: f"{v:.3f}")
    for col in cost_table.columns
}))

period      15m                                 1h                                 4h                                1d                            
metric        n typical_move_% cost_to_move      n typical_move_% cost_to_move      n typical_move_% cost_to_move     n typical_move_% cost_to_move
overall 321,262          0.391        0.768 80,292          0.764        0.393 20,063          1.452        0.207 3,349          3.520        0.085
2017     13,100          0.783        0.383  3,275          1.558        0.193    819          2.831        0.106   136          5.723        0.052
2018     34,769          0.526        0.571  8,687          1.038        0.289  2,169          1.885        0.159   363          4.368        0.069
2019     34,917          0.370        0.812  8,724          0.706        0.425  2,179          1.397        0.215   365          3.568        0.084
2020     35,045          0.407        0.738  8,754          0.804        0.373  2,187          1.452        0.20

## Step 4: Volatility of 15-minute returns by hour of day (UTC)

Standard deviation of 15-minute log returns (in %) grouped by the hour of the period start, overall and for 2017–2023 versus 2024 onward, with the number of returns in each group.

In [5]:
r15 = returns["15m"]
samples = {
    "overall": r15,
    "2017–2023": r15[r15.index < SPLIT],
    "2024 onward": r15[r15.index >= SPLIT],
}

hourly = pd.concat({
    label: r.groupby(r.index.hour).agg(std_pct=lambda x: 100 * x.std(), n="size")
    for label, r in samples.items()
}, axis=1)
hourly.index.name = "hour_utc"
print(hourly.to_string(formatters={
    col: (lambda v: f"{v:,.0f}") if col[1] == "n" else (lambda v: f"{v:.4f}")
    for col in hourly.columns
}))
print()
print("All hours:", {label: round(100 * r.std(), 4) for label, r in samples.items()})

         overall        2017–2023       2024 onward      
         std_pct      n   std_pct     n     std_pct     n
hour_utc                                                 
0         0.4198 13,398    0.4726 9,298      0.2634 4,100
1         0.3875 13,383    0.4275 9,283      0.2766 4,100
2         0.4113 13,343    0.4679 9,243      0.2383 4,100
3         0.3479 13,344    0.3948 9,244      0.2058 4,100
4         0.3465 13,353    0.3967 9,253      0.1895 4,100
5         0.3423 13,367    0.3923 9,267      0.1853 4,100
6         0.3396 13,361    0.3833 9,261      0.2098 4,100
7         0.3368 13,368    0.3814 9,268      0.2025 4,100
8         0.3592 13,377    0.4081 9,277      0.2098 4,100
9         0.3463 13,387    0.3946 9,287      0.1969 4,100
10        0.3861 13,396    0.4468 9,296      0.1859 4,100
11        0.3685 13,396    0.4224 9,296      0.1983 4,100
12        0.4164 13,394    0.4712 9,294      0.2516 4,100
13        0.4445 13,393    0.4923 9,293      0.3102 4,100
14        0.47

### Bar chart: 15-minute volatility by hour of day

In [6]:
COLORS = {"overall": "#2a78d6", "2017–2023": "#eb6834", "2024 onward": "#1baf7a"}

fig = go.Figure()
for label in samples:
    fig.add_bar(
        x=hourly.index, y=hourly[(label, "std_pct")], name=label,
        marker=dict(color=COLORS[label], line=dict(width=0)),
        hovertemplate="%{x:02d}:00 UTC<br>" + label + ": %{y:.3f}%<extra></extra>",
    )
fig.update_layout(
    title="BTC/USDT: standard deviation of 15-minute log returns by hour of day (UTC)",
    barmode="group", bargap=0.25, bargroupgap=0.08,
    xaxis=dict(title="Hour of day (UTC)", tickmode="linear", dtick=1),
    yaxis=dict(title="Std of 15-minute log returns (%)", rangemode="tozero", gridcolor="rgba(0,0,0,0.08)"),
    legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0),
    plot_bgcolor="white", height=450,
)
fig.show()

## Results

**Data.** 321,296 15-minute candles, 2017-08-17 04:00 → 2026-10-21 21:00 UTC; 565 missing candles inside the range. Returns used (dropped for a missing endpoint candle): 15m 321,262 (598); 1h 80,292 (173); 4h 20,063 (53); 1d 3,349 (2).

### Cost versus move

Typical move = standard deviation of non-overlapping log returns (%). Ratio = 0.30% round-trip cost / typical move. 2017 starts 2017-08-17; 2026 ends 2026-10-21.

| | 15m move | 15m ratio | 1h move | 1h ratio | 4h move | 4h ratio | 1d move | 1d ratio |
|---|---|---|---|---|---|---|---|---|
| overall | 0.391 | 0.768 | 0.764 | 0.393 | 1.452 | 0.207 | 3.520 | 0.085 |
| 2017 | 0.783 | 0.383 | 1.558 | 0.193 | 2.831 | 0.106 | 5.723 | 0.052 |
| 2018 | 0.526 | 0.571 | 1.038 | 0.289 | 1.885 | 0.159 | 4.368 | 0.069 |
| 2019 | 0.370 | 0.812 | 0.706 | 0.425 | 1.397 | 0.215 | 3.568 | 0.084 |
| 2020 | 0.407 | 0.738 | 0.804 | 0.373 | 1.452 | 0.207 | 4.237 | 0.071 |
| 2021 | 0.492 | 0.609 | 0.921 | 0.326 | 1.868 | 0.161 | 4.238 | 0.071 |
| 2022 | 0.342 | 0.877 | 0.681 | 0.440 | 1.312 | 0.229 | 3.371 | 0.089 |
| 2023 | 0.231 | 1.298 | 0.454 | 0.661 | 0.895 | 0.335 | 2.280 | 0.132 |
| 2024 | 0.283 | 1.060 | 0.562 | 0.534 | 1.109 | 0.271 | 2.757 | 0.109 |
| 2025 | 0.238 | 1.258 | 0.475 | 0.632 | 0.944 | 0.318 | 2.181 | 0.138 |
| 2026 | 0.233 | 1.289 | 0.453 | 0.662 | 0.876 | 0.343 | 2.353 | 0.127 |

Returns per year (n) are in the Step 3 table.

### Volatility of 15-minute returns by hour of day (UTC)

Standard deviation of 15-minute log returns (%). n per hour: overall 13,343–13,404; 2017–2023 9,243–9,308; 2024 onward 4,096–4,100.

| Hour | Overall | 2017–2023 | 2024 onward |
|---|---|---|---|
| 00 | 0.420 | 0.473 | 0.263 |
| 01 | 0.388 | 0.428 | 0.277 |
| 02 | 0.411 | 0.468 | 0.238 |
| 03 | 0.348 | 0.395 | 0.206 |
| 04 | 0.347 | 0.397 | 0.190 |
| 05 | 0.342 | 0.392 | 0.185 |
| 06 | 0.340 | 0.383 | 0.210 |
| 07 | 0.337 | 0.381 | 0.203 |
| 08 | 0.359 | 0.408 | 0.210 |
| 09 | 0.346 | 0.395 | 0.197 |
| 10 | 0.386 | 0.447 | 0.186 |
| 11 | 0.369 | 0.422 | 0.198 |
| 12 | 0.416 | 0.471 | 0.252 |
| 13 | 0.445 | 0.492 | 0.310 |
| 14 | 0.475 | 0.513 | 0.377 |
| 15 | 0.431 | 0.465 | 0.340 |
| 16 | 0.438 | 0.487 | 0.298 |
| 17 | 0.382 | 0.412 | 0.300 |
| 18 | 0.380 | 0.421 | 0.265 |
| 19 | 0.376 | 0.418 | 0.256 |
| 20 | 0.419 | 0.473 | 0.262 |
| 21 | 0.387 | 0.431 | 0.262 |
| 22 | 0.402 | 0.449 | 0.267 |
| 23 | 0.387 | 0.441 | 0.220 |
| All hours | 0.391 | 0.438 | 0.254 |

Highest hour: 14:00 in all three samples (0.475 / 0.513 / 0.377). Lowest hour: 07:00 overall (0.337), 07:00 for 2017–2023 (0.381), 05:00 for 2024 onward (0.185).

## Decision

**Decision:** Phase 9 starts with 1-hour holding periods. In 2023–2026 the 0.30% round-trip cost was 53–66% of a typical 1-hour move, so only a strong edge could cover costs; trading will therefore be selective, and Phase 9 first measures whether selective conditions (busy hours, high volatility) make 1-hour moves large enough. 15-minute holds are not researched under current cost assumptions (cost exceeded a typical 15-minute move in 2023–2026).